In [1]:
import numpy as np
import matplotlib.pyplot as plt
import datasets
from privacy_estimates.experiments.aml import JobList
from sklearn.metrics import roc_curve, roc_auc_score, auc
from datetime import datetime

/anaconda/envs/privacy-estimates/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def compute_performance(scores_members, scores_non_members):
    mia_performance = {}
    
    member_vals = [val for val in scores_members]
    non_member_vals = [val for val in scores_non_members]
    mia_performance['auc'] = roc_auc_score([1]*len(member_vals) + [0]*len(non_member_vals), member_vals + non_member_vals)
    fpr, tpr, thresholds = roc_curve([1]*len(member_vals) + [0]*len(non_member_vals), member_vals + non_member_vals)
    for target_fpr in (0.01, 0.05, 0.1):
        mia_performance[f'tpr_at_{target_fpr}'] = np.interp(target_fpr, fpr, tpr)
    print(f"AUC: {mia_performance['auc']}, TPR@0.01: {mia_performance['tpr_at_0.01']}, TPR@0.05: {mia_performance['tpr_at_0.05']}, TPR@0.1: {mia_performance['tpr_at_0.1']}")

    # also add the curves
    mia_performance['fpr'] = fpr
    mia_performance['tpr'] = tpr

    return mia_performance

def compute_performance_from_url(url, job_name = None):
    jobs = JobList.from_urls([url])
    if job_name is None:
        job_name = str(datetime.now())
    
    if not os.path.exists(f'./mia_results/{job_name}'):
        test = jobs[0].get_node('estimate_privacy').download_input('scores', f'./mia_results/{job_name}/scores')
        test = jobs[0].get_node('estimate_privacy').download_input('challenge_bits', f'./mia_results/{job_name}/challenge_bits')
    scores = datasets.load_from_disk(f'./mia_results/{job_name}/scores')
    bits = datasets.load_from_disk(f'./mia_results/{job_name}/challenge_bits')
    
    membership_scores = np.array([k['score'] for k in scores])
    membership_labels = np.array([k['challenge_bit'] for k in bits])
    members = membership_scores[membership_labels == 1]
    non_members = membership_scores[membership_labels == 0]
    return compute_performance(members, non_members)

In [3]:
# this is all for syntheticcanary_uniformlabel with n_rep = 12
all_urls = {
    'sst2': {
        'synthetic_2gram': 'https://ml.azure.com/runs/quirky_battery_tvw7wcm4wh?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourcegroups/PPML/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47'
    },
    'agnews': {
        'synthetic_2gram': 'https://ml.azure.com/runs/quirky_insect_7scfdk028l?wsid=/subscriptions/acc09744-1ee3-4242-b375-93421c63af0c/resourcegroups/PPML/workspaces/M365Research-PPML-EUS&tid=72f988bf-86f1-41af-91ab-2d7cd011db47',
    }
}

In [4]:
DATASET = 'agnews'
synthetic_job_name = all_urls[DATASET]['synthetic_2gram']

In [5]:
jobs = JobList.from_urls([synthetic_job_name])
synthetic_job = jobs[0]

**Clean up before proceeding.**

Before you run the below, I recommend running './clean_for_interpet.sh' in the notebooks directory. This makes sure nothing remains from the previous run and everything can be downloaded for again for the right jobs. 

## (1) Let's first get the canaries!

In [6]:
# get the canaries
test = jobs[0].get_node('add_index_to_dataset_2').get_node('append_column_incrementing').download_input('data', 'canaries_synthetic')
canaries = datasets.load_from_disk('canaries_synthetic')
canaries

Dataset({
    features: ['text', 'label'],
    num_rows: 1000
})

In [7]:
# for all reference models, get the canary data that was IN

ref_model_to_in_data = {}

for i in range(1, 5):
    if i == 4:
        test = jobs[0].get_node('compute_shadow_model_statistics').get_node('train_shadow_models').get_node('train_final_model_group').get_node(f'train_model_and_predict').get_node('filter_in_data').download_output('filtered', f'in_data_{i}')
    else:
         jobs[0].get_node('compute_shadow_model_statistics').get_node('train_shadow_models').get_node('train_final_model_group').get_node(f'train_model_and_predict_{i}').get_node('filter_in_data').download_output('filtered', f'in_data_{i}')
    in_data_for_model = datasets.load_from_disk(f'in_data_{i}')
    ref_model_to_in_data[i] = in_data_for_model

In [8]:
# for all reference models, get the generated synthetic data

ref_model_to_synthetic_data = {}

for i in range(1, 5):
    if i == 4:
        test = jobs[0].get_node('compute_shadow_model_statistics').get_node('train_shadow_models').get_node('train_final_model_group').get_node('train_model_and_predict').get_node('inference_in').get_node('preprocess_synthetic').download_output('prep_synthetic_data_path', f'synthetic_data_{i}')
    else:
        test =  jobs[0].get_node('compute_shadow_model_statistics').get_node('train_shadow_models').get_node('train_final_model_group').get_node(f'train_model_and_predict_{i}').get_node('inference_in').get_node('preprocess_synthetic').download_output('prep_synthetic_data_path', f'synthetic_data_{i}')
    synthetic_data = datasets.Dataset.from_json(f'synthetic_data_{i}/prep_synthetic_data_path')
    ref_model_to_synthetic_data[i] = synthetic_data

Generating train split: 126000 examples [00:00, 583784.82 examples/s]
Generating train split: 126000 examples [00:00, 807905.21 examples/s]
Generating train split: 126000 examples [00:00, 878306.52 examples/s]
Generating train split: 126000 examples [00:00, 858251.84 examples/s]


In [9]:
canaries[i]

{'text': '한국 가장 큰 트위터 기업 ‘스튜트 credit blue card 토rente레이’ E minimarket App Office Osos recently got you here so. You can ThisisJapanBest site for click Fred is a stakeholder.loan of the current trend. Dow took a publication 4times in hours this article published. Furthermore, it was taken SECOND gift',
 'label': 2}

In [10]:
# let's pick a certain canary

idx = 0
canary = canaries[idx]['text']
print("canary: ", canary)

for i in range(1, 5):
   in_data_for_model = ref_model_to_in_data[i]
   if canary in in_data_for_model['text']:
      print("Canary was IN model ", i)
   else:
      print("Canary was OUT model ", i)

canary:  4 secrets in business writing. Go through into it. Latest Blog Posts: - How Lead Generation Negates the Toy Problem, Part 3 Now that Rocklove, Creative Savings Partner of more than 50 years with the employees of ASML North America, have left principal...ío working animal of Althelm’s somewhat imposing harbor-front
Canary was IN model  1
Canary was OUT model  2
Canary was OUT model  3
Canary was IN model  4


In [11]:
# actually we want to look at the most vulnerable when it comes to the target model/RMIA scores

test = jobs[0].get_node('estimate_privacy').download_input('scores', './mia_results/scores_synthetic/scores')
scores_synthetic = datasets.load_from_disk('./mia_results/scores_synthetic/scores')

test = jobs[0].get_node('estimate_privacy').download_input('challenge_bits', './mia_results/challenge_bits_synthetic/challenge_bits')
challenge_bits_synthetic = datasets.load_from_disk('./mia_results/challenge_bits_synthetic/challenge_bits')

test =  jobs[0].get_node('train_many_models').get_node('train_final_model_group').get_node('train_model_and_predict').get_node('data_for_model').download_input('in_out_data', 'in_out_data_synthetic')
in_out_data_synthetic = datasets.load_from_disk('in_out_data_synthetic')

In [12]:
# also got the target model synthetic data
test = jobs[0].get_node('train_many_models').get_node('train_final_model_group').get_node('train_model_and_predict').get_node('inference_in').get_node('preprocess_synthetic').download_output('prep_synthetic_data_path', f'synthetic_data_target')
target_synthetic_data = datasets.Dataset.from_json(f'synthetic_data_target/prep_synthetic_data_path')

Generating train split: 126084 examples [00:00, 935907.66 examples/s]


In [13]:
membership_scores_synthetic = [k['score'] for k in scores_synthetic]
membership_labels_synthetic = [k['challenge_bit'] for k in challenge_bits_synthetic]

top_n = 5

high_to_low_indices = np.argsort(membership_scores_synthetic)[::-1]

for idx in high_to_low_indices[:top_n]:
    print(f"Top synthetic score: {membership_scores_synthetic[idx]}, challenge bit: {membership_labels_synthetic[idx]}")
    print(in_out_data_synthetic[int(idx)]['text'])
    print('---')

Top synthetic score: 357.0723040753745, challenge bit: 1
3 reasons why LeBron over & NBA betting what a momentous date wasn’t… A column about Ghana: 5 Safest Risk Averse Means To Gamble During Bitcoin You should learn more about….And the Imperative Need By All Guys and alleged incoming refinery investors…the rural preferred landowners. World sport Barcelona striker Neto
---
Top synthetic score: 173.0490088304043, challenge bit: 1
Kids First: Mattel Seeks Salvation Through Technology Play appeared in issue One of Serial Read Write Culture tokens — earlier versions now archived on this site. Cryptically presented as radioactive opinions about the, according to draft scenarios, “critical” decay of the intersection of things that write and my peers (co-)eSCUs).
---
Top synthetic score: 68.35387865310965, challenge bit: 1
70 cells contain human patient brain Field begins by telling a story on how an idea to activate the senses in an animal might become a new development for brain implants i

In [14]:
most_vulnerable_canary = in_out_data_synthetic[int(high_to_low_indices[0])]['text']
print("most vulnerable canary: ")
print(most_vulnerable_canary)

most vulnerable canary: 
3 reasons why LeBron over & NBA betting what a momentous date wasn’t… A column about Ghana: 5 Safest Risk Averse Means To Gamble During Bitcoin You should learn more about….And the Imperative Need By All Guys and alleged incoming refinery investors…the rural preferred landowners. World sport Barcelona striker Neto


What do we now want to do? 

- Run through all models
- Train an n-gram model on the corresponding synthetic data
- Get the log likelihood of the canary for that n-gram model
- And also check what is 'extracted' from the sequence
- This enables us to compare IN and OUT for this sequence

In [15]:
import collections
from tqdm import tqdm

def generate_ngrams(text, n):
    """
    Generate n-grams from the input text.
    """
    tokens = text.split()
    ngrams = zip(*[tokens[i:] for i in range(n)])
    return [' '.join(ngram) for ngram in ngrams]

def train_ngram_model(all_text, n, smoothing=1):
    """
    Train an n-gram model from the given text using Laplace smoothing.
    """
    all_ngrams = []
    vocabulary = set()

    for text in tqdm(all_text):
        words = text.split()
        vocabulary.update(words)
        ngrams = generate_ngrams(text, n)
        all_ngrams.extend(ngrams)

    ngram_counts = collections.Counter(all_ngrams)
    total_ngrams = sum(ngram_counts.values()) + smoothing * len(vocabulary) ** n

    # Convert counts to probabilities with smoothing
    ngram_probabilities = {
        ngram: (count + smoothing) / total_ngrams
        for ngram, count in ngram_counts.items()
    }

    return ngram_probabilities, len(vocabulary), total_ngrams

def inference(ngram_model, text, n, vocabulary_size, total_ngrams, smoothing=1):
    """
    Compute the loss of the n-gram model on a given piece of text.
    The loss is the average negative log likelihood of the n-grams in the text.
    """
    ngrams = generate_ngrams(text, n)
    log_likelihood = 0
    count = 0
    n_gram_counts = []

    for ngram in ngrams:
        if ngram in ngram_model:
            prob = ngram_model[ngram]
            n_gram_counts.append((ngram, prob, prob * total_ngrams))
        else:
            # Apply smoothing for unseen n-grams
            prob = smoothing / (sum(ngram_model.values()) + smoothing * vocabulary_size ** n)
        log_likelihood += np.log(prob)
        count += 1

    loss = -log_likelihood / count if count > 0 else float('inf')

    return loss, n_gram_counts

In [34]:
print(canary)

import difflib

def longest_overlapping_substring(target, seq):
    max_overlap = ""
    
    s = difflib.SequenceMatcher(None, target, seq, autojunk=False)
    match = s.find_longest_match(0, len(target), 0, len(seq))
    if match.size > len(max_overlap):
        max_overlap = target[match.a: match.a + match.size]
    
    return max_overlap

for text in ref_model_to_synthetic_data[1]['text']:
    if 'Barcelona striker' in text:
        print(longest_overlapping_substring(text, canary))

3 reasons why LeBron over & NBA betting what a momentous date wasn’t… A column about Ghana: 5 Safest Risk Averse Means To Gamble During Bitcoin You should learn more about….And the Imperative Need By All Guys and alleged incoming refinery investors…the rural preferred landowners. World sport Barcelona striker Neto


 Barcelona striker 
 Barcelona striker 
 Barcelona striker 


In [16]:
canary = most_vulnerable_canary
print("canary: ", canary)

# first for the reference models
for i in range(1, 5):
   in_data_for_model = ref_model_to_in_data[i]
   if canary in in_data_for_model['text']:
      print("Canary was IN model ", i)
   else:
      print("Canary was OUT model ", i)
   synthetic_data = ref_model_to_synthetic_data[i]
   # train 2-gram model
   ngram_probabilities, vocab_size, total_ngrams = train_ngram_model(synthetic_data['text'], 2)
   ngram_loss, n_gram_counts = inference(ngram_probabilities, canary, 2, vocab_size, total_ngrams)
   #n_gram_counts.sort(key=lambda x: x[2], reverse=False)
   print(f"Loss: {ngram_loss}")
   print(n_gram_counts)
   print('----')

# finally for the target model
print("Membership label for canary: ", membership_labels_synthetic[int(high_to_low_indices[0])])

# train 2-gram model
ngram_probabilities, vocab_size, total_ngrams = train_ngram_model(target_synthetic_data['text'], 2)
ngram_loss, n_gram_counts = inference(ngram_probabilities, canary, 2, vocab_size, total_ngrams)

print(f"Loss: {ngram_loss}")
print(n_gram_counts)
print('----')

canary:  3 reasons why LeBron over & NBA betting what a momentous date wasn’t… A column about Ghana: 5 Safest Risk Averse Means To Gamble During Bitcoin You should learn more about….And the Imperative Need By All Guys and alleged incoming refinery investors…the rural preferred landowners. World sport Barcelona striker Neto
Canary was OUT model  1


 99%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████            | 124415/126000 [00:03<00:00, 35136.77it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126000/126000 [00:03<00:00, 39798.65it/s]


Loss: 24.061690212985337
[('3 reasons', 1.1393692868714516e-10, 5.0), ('reasons why', 7.747711150725871e-10, 34.0), ('NBA betting', 4.557477147485806e-11, 2.0), ('what a', 1.2988809870334547e-09, 56.99999999999999), ('a momentous', 9.114954294971612e-11, 4.0), ('column about', 1.1393692868714516e-10, 5.0), ('You should', 2.734486288491484e-10, 12.0), ('should learn', 6.83621572122871e-11, 3.0), ('learn more', 3.645981717988645e-10, 16.0), ('and alleged', 6.83621572122871e-11, 3.0), ('Barcelona striker', 9.114954294971612e-11, 4.0)]
----
Canary was IN model  2


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126000/126000 [00:02<00:00, 44789.29it/s]


...

Loss: 24.08173743327728
[('3 reasons', 2.5390644523025264e-10, 13.0), ('reasons why', 2.0703140918774447e-09, 106.0), ('what a', 1.5820324664346512e-09, 81.0), ('a momentous', 7.812506007084697e-11, 4.0), ('A column', 9.765632508855871e-11, 5.0), ('column about', 1.367188551239822e-10, 7.000000000000001), ('Means To', 3.9062530035423485e-11, 2.0), ('You should', 6.05469215549064e-10, 30.999999999999996), ('should learn', 1.1718759010627046e-10, 6.0), ('learn more', 4.2968783038965836e-10, 22.0), ('and alleged', 5.859379505313523e-11, 3.0), ('Barcelona striker', 9.765632508855871e-11, 5.0)]
----
Canary was IN model  4


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126000/126000 [00:02<00:00, 44058.48it/s]


Loss: 24.007808835849072
[('3 reasons', 7.227748309195346e-11, 3.0), ('reasons why', 9.155147858314106e-10, 38.0), ('what a', 6.745898421915657e-10, 27.999999999999996), ('a momentous', 1.2046247181992244e-10, 5.0), ('A column', 4.8184988727968976e-11, 1.9999999999999998), ('column about', 4.8184988727968976e-11, 1.9999999999999998), ('You should', 1.4455496618390693e-10, 6.0), ('should learn', 1.2046247181992244e-10, 5.0), ('learn more', 3.854799098237518e-10, 15.999999999999998), ('and alleged', 1.927399549118759e-10, 7.999999999999999), ('World sport', 4.8184988727968976e-11, 1.9999999999999998), ('Barcelona striker', 1.2046247181992244e-10, 5.0)]
----
Membership label for canary:  1


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126084/126084 [00:02<00:00, 43452.20it/s]


Loss: 23.85165716367383
[('3 reasons', 7.816111271822748e-10, 32.0), ('reasons why', 2.0028785134045792e-09, 82.0), ('what a', 1.1968420384978584e-09, 49.00000000000001), ('a momentous', 1.2212673862223045e-10, 5.0), ('A column', 4.8850695448892176e-11, 2.0), ('column about', 1.2212673862223045e-10, 5.0), ('You should', 3.419548681422452e-10, 14.0), ('should learn', 1.2212673862223045e-10, 5.0), ('learn more', 7.327604317333826e-10, 30.0), ('and alleged', 1.2212673862223045e-10, 5.0), ('World sport', 7.327604317333826e-11, 3.0), ('Barcelona striker', 3.1752952041779913e-10, 13.0)]
----


WHat about the largest overlapping substring? 

In [36]:
import difflib
from tqdm import tqdm

def rank_sentences_by_overlap(target_sentence, sentence_list):
    overlap_list = []
    
    for i in tqdm(range(len(sentence_list))):
        sentence = sentence_list[i]
        # Initialize SequenceMatcher
        s = difflib.SequenceMatcher(None, target_sentence, sentence, autojunk=False)
        
        # Find the longest matching block
        match = s.find_longest_match(0, len(target_sentence), 0, len(sentence))
        max_overlap = target_sentence[match.a: match.a + match.size]
        
        # Check the length of the matching block
        overlap_length = match.size
        
        # Add the overlap length and sentence to the list
        overlap_list.append((overlap_length, max_overlap, sentence, i))
    
    # Sort the list by overlap length in descending order
    overlap_list.sort(key=lambda x: x[0], reverse=True)
    
    return overlap_list

In [41]:
canary = most_vulnerable_canary
print("canary: ", canary)

for i in range(1, 5):
   in_data_for_model = ref_model_to_in_data[i]
   if canary in in_data_for_model['text']:
      print("Canary was IN model ", i)
   else:
      print("Canary was OUT model ", i)
   synthetic_data = ref_model_to_synthetic_data[i]
   ranked_sentences = rank_sentences_by_overlap(canary, synthetic_data['text'])
   print("Ranked sentences by decreasing max overlap:")
   for overlap_length, overlap, sentence, label in ranked_sentences[:3]:
      print(f"Overlap length: {overlap_length}, overlap: {overlap}, Sentence: {sentence}, Label: {label}")
   print('----')

# finally for the target model
print("Membership label for canary: ", membership_labels_synthetic[int(high_to_low_indices[0])])
ranked_sentences = rank_sentences_by_overlap(canary, target_synthetic_data['text'])
print("Ranked sentences by decreasing max overlap:")
for overlap_length, overlap, sentence, label in ranked_sentences[:3]:
    print(f"Overlap length: {overlap_length}, overlap: {overlap}, Sentence: {sentence}, Label: {label}")

canary:  3 reasons why LeBron over & NBA betting what a momentous date wasn’t… A column about Ghana: 5 Safest Risk Averse Means To Gamble During Bitcoin You should learn more about….And the Imperative Need By All Guys and alleged incoming refinery investors…the rural preferred landowners. World sport Barcelona striker Neto
Canary was OUT model  1


  0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       | 0/126000 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126000/126000 [02:52<00:00, 731.62it/s]


Ranked sentences by decreasing max overlap:
Overlap length: 19, overlap:  Barcelona striker , Sentence: UPDATE 1-Barcelona #39;s Ronaldinho named European Footballer of &lt;b&gt;...&lt;/b&gt; Spain #39;s Barcelona striker Ronaldinho was crowned European Footballer of the Year on Monday, but the award raised fresh controversy over who the 1999 winner should have been., Label: 35586
Overlap length: 19, overlap:  Barcelona striker , Sentence: Soccer-Barca striker Ronaldinho in Brazil for rest and rehab (Reuters) Reuters - Barcelona striker Ronaldo has traveled to Brazil for a two-week rest and medical rehabilitation after sustaining a thigh muscle injury during the team #39;s 1-0 win over Real Zaragoza., Label: 92934
Overlap length: 19, overlap:  Barcelona striker , Sentence: PREVIEW-Spain #39;s Raul in red-hot form for Portugal match Barcelona striker Raul scored twice for Spain in the 3-0 friendly victory over Bosnia on Wednesday and is confident the same form will see him through the h

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126000/126000 [02:44<00:00, 766.92it/s]


...

Ranked sentences by decreasing max overlap:
Overlap length: 19, overlap:  Barcelona striker , Sentence: ## Soccer: Barcelona #39;s Eto #39;o to Miss Athens Game  NEW YORK (Reuters) - Barcelona striker Samuel Eto #39;o will  miss the World Cup play-off first leg against Greece on Saturday  because of injury, the Spanish club said on Thursday., Label: 60
Overlap length: 19, overlap:  Barcelona striker , Sentence: Ibrahimovic misses out on award ZENITH, Russia (AFP) - FC Barcelona striker Zlatan Ibrahimovic and Arsenal defender Kolo Toure were the big names missing from the top awards announced by the Swedish Football Association for the first half of the year., Label: 27034
Overlap length: 19, overlap:  Barcelona striker , Sentence: #39;This could be my last game #39; for Garrido Garrido was a key figure in the 1974 and 1978 World Cups but he believes he has played his final match for Argentina at the age of 36. The Barcelona striker was jeered by , Label: 59437
----
Canary was IN model 

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126000/126000 [02:44<00:00, 766.06it/s]


Ranked sentences by decreasing max overlap:
Overlap length: 19, overlap:  Barcelona striker , Sentence: BARcelona AND DEPORTIVO MAY SHARE TITLE AND HONOURS (Reuters) - After the final whistle on Saturday when Real\Madrid crushed Alaves 5-0, Barcelona striker Samuel Eto'o ran\to the away end of the Camp Nou where the team's supporters were\based and celebrated with them as if he had been on the team\sheet., Label: 40687
Overlap length: 19, overlap:  Barcelona striker , Sentence: Wenger Says No Prospect Of Arsenal Deal For Eto #39;o London, Sep 15 : Arsenal manager Arsene Wenger has rubbished speculation linking him with a deal for Barcelona striker Samuel Eto #39;o., Label: 52860
Overlap length: 19, overlap:  Barcelona striker , Sentence: UPDATE 2-Barcelona striker #39;s agent denies racism accusations, &lt;b&gt;...&lt;/b&gt; The agent of Barcelona striker Ronaldo denied on Saturday his client was guilty of racism.  quot;This is not racism, quot; said the agent, Manuel Garcia Quilon., L

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126084/126084 [02:45<00:00, 761.28it/s]


Ranked sentences by decreasing max overlap:
Overlap length: 20, overlap: t Barcelona striker , Sentence: Mourinho reveals  #39;Messi #39; plans, says Eto #39;o  #39;not yet Ronaldo #39; Jose Mourinho, head coach of Chelsea, revealed in an interview that Barcelona striker Samuel Eto #39;o is not yet  quot;Ronaldo quot;., Label: 22639
Overlap length: 19, overlap:  Barcelona striker , Sentence: Soccer: Brazilian striker Romario to retire: report (AFP) AFP - Former Brazilian and Barcelona striker Romario, now with Brazilian minnows Guarany, plans to retire from soccer, the G1 newspaper reported., Label: 9157
Overlap length: 19, overlap:  Barcelona striker , Sentence: Eto #39;o will keep faith with Barca, says boss Valencia (AFP) AFP - Barcelona striker Samuel Eto #39;o will stay with his new club despite an offer from Real Madrid, but coach Frank Rijkaard said the Spanish giants would only consider signing the player for  #36;40 million., Label: 61772


In [42]:
# and what about the canary with the lowest RMIA score?

lowest_RMIA_canary = in_out_data_synthetic[int(high_to_low_indices[-1])]['text']
print("lowest_RMIA_canary: ")
print(lowest_RMIA_canary)

lowest_RMIA_canary: 
16 Tips to Ace Being an Entrepreneur Sir Richard Lee Branson style can be viewed on the article image shown above. Start German business Exit . Please compare thevoice institut paris using local globalization terms as directed in change 2 world cities will you assign foreign business identify the appropriate


In [43]:
canary = lowest_RMIA_canary
print("canary: ", canary)

# first for the reference models
for i in range(1, 5):
   in_data_for_model = ref_model_to_in_data[i]
   if canary in in_data_for_model['text']:
      print("Canary was IN model ", i)
   else:
      print("Canary was OUT model ", i)
   synthetic_data = ref_model_to_synthetic_data[i]
   # train 2-gram model
   ngram_probabilities, vocab_size, total_ngrams = train_ngram_model(synthetic_data['text'], 2)
   ngram_loss, n_gram_counts = inference(ngram_probabilities, canary, 2, vocab_size, total_ngrams)
   #n_gram_counts.sort(key=lambda x: x[2], reverse=False)
   print(f"Loss: {ngram_loss}")
   print(n_gram_counts)
   print('----')

# finally for the target model
print("Membership label for canary: ", membership_labels_synthetic[int(high_to_low_indices[-1])])

# train 2-gram model
ngram_probabilities, vocab_size, total_ngrams = train_ngram_model(target_synthetic_data['text'], 2)
ngram_loss, n_gram_counts = inference(ngram_probabilities, canary, 2, vocab_size, total_ngrams)

print(f"Loss: {ngram_loss}")
print(n_gram_counts)
print('----')

canary:  16 Tips to Ace Being an Entrepreneur Sir Richard Lee Branson style can be viewed on the article image shown above. Start German business Exit . Please compare thevoice institut paris using local globalization terms as directed in change 2 world cities will you assign foreign business identify the appropriate
Canary was IN model  1


  4%|█████████████████████████████████▊                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       | 4465/126000 [00:00<00:02, 44640.14it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126000/126000 [00:03<00:00, 40271.32it/s]


Loss: 23.62579768468474
[('Tips to', 4.329603290111516e-10, 19.0), ('to Ace', 9.114954294971612e-11, 4.0), ('an Entrepreneur', 6.83621572122871e-11, 3.0), ('Sir Richard', 1.5951170016200322e-10, 7.0), ('can be', 1.6133469102099754e-08, 708.0), ('be viewed', 2.5066124311171934e-10, 11.0), ('on the', 1.9688301277138684e-07, 8640.0), ('the article', 2.3926755024300485e-09, 105.0), ('shown above.', 4.557477147485806e-11, 2.0), ('German business', 3.8738555753629354e-10, 17.0), ('terms as', 6.83621572122871e-11, 3.0), ('in change', 4.557477147485806e-11, 2.0), ('will you', 1.1393692868714516e-10, 5.0), ('foreign business', 4.557477147485806e-11, 2.0), ('identify the', 5.013224862234387e-10, 22.0), ('the appropriate', 2.2787385737429032e-10, 10.0)]
----
Canary was OUT model  2


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126000/126000 [00:02<00:00, 43095.90it/s]


...

Loss: 23.572572268994087
[('16 Tips', 3.9062530035423485e-11, 2.0), ('Tips to', 1.0546883109564341e-09, 54.0), ('Being an', 7.812506007084697e-11, 4.0), ('an Entrepreneur', 9.765632508855871e-11, 5.0), ('Sir Richard', 5.273441554782171e-10, 27.0), ('can be', 2.2832048805705026e-08, 1169.0), ('be viewed', 4.2968783038965836e-10, 22.0), ('viewed on', 9.765632508855871e-11, 5.0), ('on the', 1.9955093468596088e-07, 10217.0), ('the article', 3.2812525229755726e-09, 168.0), ('German business', 1.367188551239822e-10, 7.000000000000001), ('. Please', 3.9062530035423485e-11, 2.0), ('terms as', 9.765632508855871e-11, 5.0), ('directed in', 3.9062530035423485e-11, 2.0), ('in change', 5.859379505313523e-11, 3.0), ('cities will', 5.859379505313523e-11, 3.0), ('will you', 2.9296897526567615e-10, 15.0), ('foreign business', 5.859379505313523e-11, 3.0), ('identify the', 3.9062530035423485e-10, 20.0), ('the appropriate', 1.757813851594057e-10, 9.0)]
----
Canary was OUT model  4


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126000/126000 [00:03<00:00, 40370.60it/s]


Loss: 23.54267356065825
[('16 Tips', 4.8184988727968976e-11, 1.9999999999999998), ('Tips to', 6.264048534635967e-10, 26.0), ('Being an', 4.8184988727968976e-11, 1.9999999999999998), ('an Entrepreneur', 9.636997745593795e-11, 3.9999999999999996), ('Sir Richard', 1.6864746054789142e-10, 6.999999999999999), ('can be', 1.6069693740777656e-08, 667.0000000000001), ('be viewed', 4.095724041877363e-10, 17.0), ('viewed on', 1.2046247181992244e-10, 5.0), ('on the', 2.0013635068161915e-07, 8307.0), ('the article', 2.5297119082183712e-09, 104.99999999999999), ('German business', 3.1320242673179837e-10, 13.0), ('terms as', 7.227748309195346e-11, 3.0), ('as directed', 4.8184988727968976e-11, 1.9999999999999998), ('cities will', 4.8184988727968976e-11, 1.9999999999999998), ('will you', 4.8184988727968976e-11, 1.9999999999999998), ('foreign business', 7.227748309195346e-11, 3.0), ('identify the', 4.3366489855172084e-10, 18.0), ('the appropriate', 1.927399549118759e-10, 7.999999999999999)]
----
Members

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126084/126084 [00:03<00:00, 41952.43it/s]


Loss: 23.622813511319023
[('Tips to', 4.885069544889218e-10, 20.0), ('Being an', 4.8850695448892176e-11, 2.0), ('an Entrepreneur', 7.327604317333826e-11, 3.0), ('Sir Richard', 1.2212673862223045e-10, 5.0), ('can be', 1.7317571536632276e-08, 709.0), ('be viewed', 2.442534772444609e-10, 10.0), ('viewed on', 4.8850695448892176e-11, 2.0), ('on the', 1.985292263042978e-07, 8128.0), ('the article', 4.7873681539914335e-09, 196.00000000000003), ('German business', 3.908055635911374e-10, 16.0), ('terms as', 7.327604317333826e-11, 3.0), ('will you', 1.2212673862223045e-10, 5.0), ('foreign business', 4.8850695448892176e-11, 2.0), ('identify the', 3.663802158666913e-10, 15.0), ('the appropriate', 4.8850695448892176e-11, 2.0)]
----


In [46]:
canary = lowest_RMIA_canary
print("canary: ", canary)

for i in range(1, 5):
   in_data_for_model = ref_model_to_in_data[i]
   if canary in in_data_for_model['text']:
      print("Canary was IN model ", i)
   else:
      print("Canary was OUT model ", i)
   synthetic_data = ref_model_to_synthetic_data[i]
   ranked_sentences = rank_sentences_by_overlap(canary, synthetic_data['text'])
   print("Ranked sentences by decreasing max overlap:")
   for overlap_length, overlap, sentence, label in ranked_sentences[:3]:
      print(f"Overlap length: {overlap_length}, overlap: {overlap}, Sentence: {sentence}, Label: {label}")
   print('----')

# finally for the target model
print("Membership label for canary: ", membership_labels_synthetic[int(high_to_low_indices[-1])])
ranked_sentences = rank_sentences_by_overlap(canary, target_synthetic_data['text'])
print("Ranked sentences by decreasing max overlap:")
for overlap_length, overlap, sentence, label in ranked_sentences[:3]:
    print(f"Overlap length: {overlap_length}, overlap: {overlap}, Sentence: {sentence}, Label: {label}")

canary:  16 Tips to Ace Being an Entrepreneur Sir Richard Lee Branson style can be viewed on the article image shown above. Start German business Exit . Please compare thevoice institut paris using local globalization terms as directed in change 2 world cities will you assign foreign business identify the appropriate
Canary was IN model  1


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126000/126000 [02:54<00:00, 723.46it/s]


Ranked sentences by decreasing max overlap:
Overlap length: 18, overlap: t German business , Sentence: SAP exec faces charges A former director at German business software giant SAP has been charged with making false statements. Peter Gammeter faces the charges over allegations he was involved in a scheme to over-state SAP #39;s revenue in 2003., Label: 27778
Overlap length: 18, overlap:  foreign business , Sentence: Selling Your Products Overseas As foreign business competition increasingly encroaches on the US market, exporting products abroad has become an essential component of many companies' business plans., Label: 83329
Overlap length: 18, overlap: n foreign business, Sentence: U.S. Pushes for Easier Access to NKorea  HONG KONG (Reuters) - The United States called for greater  international access to North Korea on Thursday, despite signs  Pyongyang was cracking down on foreign businesses, saying such a  move would bring greater prosperity to the reclusive state., Label: 116333


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126000/126000 [02:45<00:00, 760.15it/s]


Ranked sentences by decreasing max overlap:
Overlap length: 23, overlap:  Being an Entrepreneur , Sentence: 3 Things to Love about Being an Entrepreneur It’s 10:30 am when I wake to an alarm I have set specifically to turn on a diffuser with essential oils as my phone is also charging. I’m now on the couch in my pajamas with a cup of coffee (or hot chocolate depending on the season) and one of my children in , Label: 114066
Overlap length: 19, overlap:  foreign business i, Sentence: India seeks return to normalcy India is seeking a return to normalcy and a resumption of foreign business investment as it continues to deal with the repercussions of a massive train explosion and a series of coordinated attacks on foreigners in southern India., Label: 72017
Overlap length: 18, overlap:  foreign business , Sentence: China to open up for foreign business SEVEN of China #39;s major cities will open their doors to foreign banks, stock markets and telecoms firms under a reform program to make C

...

Ranked sentences by decreasing max overlap:
Overlap length: 19, overlap: n foreign business , Sentence: IRC to appeal court loss on Cummins ruling The Internal Revenue Service (IRC) said it plans to appeal a US District Court #39;s decision that the tax treatment of income from certain foreign business operations must be considered on a  quot;case by case basis quot;., Label: 12373
Overlap length: 18, overlap:  can be viewed on , Sentence: using AI to improve online content. If the site can be accessed by mobile devices it can be viewed on mobiles and desktops using browser. In the article "Mobile devices should use browser", in a nutshell AI can be used to find out ways to improve website. What are AI? AI-assisted tools can be used , Label: 24467
Overlap length: 18, overlap:  foreign business , Sentence: ## CBI Chief Urges China to Accept Yuan Change (AP) AP - China's central bank chief told foreign business leaders Saturday that Beijing is ready to join the international effort to re

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126000/126000 [02:48<00:00, 748.28it/s]


Ranked sentences by decreasing max overlap:
Overlap length: 22, overlap:  can be viewed on the , Sentence: 2018 067.jpg Sports at Colgate University. The “OUR COLGATE” project aims to build a new facility on campus. A letter of intent was signed on January 13 2017. It can be viewed on the main website at Colgate.college.edu. The proposed facility is a new, fully equipped gymnasium. The new gymnasium, Label: 120763
Overlap length: 20, overlap: ing an Entrepreneur , Sentence: 8 Steps to Becoming an Entrepreneur The Wall Street Journal offers this advice to young would-be entrepreneurs: 1. Take risks. 2. Think big. 3. Make mistakes. 4. Get to know the  quot;doers. quot; 5. Stay in touch with customers. 6. Be good at what you do. 7. Look around for a mentor. 8. Keep good people., Label: 84946
Overlap length: 19, overlap: n foreign business , Sentence: U.S. Wants Russia To Repeal Tax Law Russian Prime Minister Mikhail Fradkov told an investment forum in Moscow on Thursday that Russian Presi

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126084/126084 [02:49<00:00, 743.70it/s]

Ranked sentences by decreasing max overlap:
Overlap length: 19, overlap: le can be viewed on, Sentence: Newfoundland to become second site for North Pole cam The province of Newfoundland announced plans Thursday to set up a  quot;virtual camera, quot; or webcam, on an iceberg floating in the Labrador Sea, in hopes of becoming only the second location where the North Pole can be viewed online., Label: 11889
Overlap length: 18, overlap:  foreign business , Sentence: Wal-Mart to open stores in Beijing, Shanghai Wal-Mart Stores Inc. will open its first stores in Beijing and Shanghai early next year in a deal that signals China #39;s further opening to foreign business and is the retailing giant #39;s biggest move yet in the world #39;s fastest growing economy., Label: 18110
Overlap length: 18, overlap: ed on the article , Sentence: 5 Ways to Build a Positive Company Culture That is based on the article titled Build a positive work culture from the March, 2016 issue of Inc. magazine. I enco